In [10]:
reticulate::use_python("/home/cstansbu/miniconda3/envs/r_scminer/bin/python", required = TRUE)
reticulate::py_config()

python:         /home/cstansbu/miniconda3/envs/r_scminer/bin/python
libpython:      /home/cstansbu/miniconda3/envs/r_scminer/lib/libpython3.9.so
pythonhome:     /home/cstansbu/miniconda3/envs/r_scminer:/home/cstansbu/miniconda3/envs/r_scminer
version:        3.9.18 | packaged by conda-forge | (main, Dec 23 2023, 16:33:10)  [GCC 12.3.0]
numpy:          /home/cstansbu/miniconda3/envs/r_scminer/lib/python3.9/site-packages/numpy
numpy_version:  2.0.2

NOTE: Python version was forced by use_python() function

In [11]:
suppressPackageStartupMessages(library("scMINER"))
suppressPackageStartupMessages(library("anndata"))
suppressPackageStartupMessages(library("SingleCellExperiment"))
suppressPackageStartupMessages(library("Matrix"))

In [12]:
reticulate::py_module_available("anndata")  # should return TRUE

[1] TRUE

# Project path

In [13]:
project_path <- "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/scMINER"
project_name <- "test"
full_project_path <- file.path(project_path, project_name)

if (dir.exists(full_project_path)) {
  message("Project directory already exists at: ", full_project_path)
} else {
  scminer_dir <- createProjectSpace(
    project_dir = project_path, 
    project_name = project_name,
  )
  message("Project space created at: ", scminer_dir)
}


Project directory already exists at: /nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/scMINER/test



# Load Data

In [14]:
file_path <- "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/pseudotime.h5ad"

adata <- read_h5ad(file_path)
adata

AnnData object with n_obs × n_vars = 15867 × 21412
    obs: 'batch', 'phase', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_counts', 'n_genes', 'n_reads', 'raw_clusters', 'bbknn_clusters', 'harmony_clusters', 'cluster_str', 'barcoded_phase', 'S_score', 'G2M_score', 'dpt_pseudotime', 'dpt_groups', 'dpt_order', 'dpt_order_indices', 'G1_pseudotime', 'G1_order', 'G2M_pseudotime', 'G2M_order', 'mean_pseudotime', 'mean_order', 'nnz'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'highly_variable', 'means', 'dispersions', 'dispersio

# Create SparseEset object

In [15]:
input_matrix <- adata$layers['raw_counts']

# Print dimensions
cat("Matrix dimensions:\n")
cat("Rows (genes):", nrow(input_matrix), "\n")
cat("Columns (cells):", ncol(input_matrix), "\n")

# Print class
cat("Matrix class:", class(input_matrix), "\n")

Matrix dimensions:
Rows (genes): 15867 
Columns (cells): 21412 
Matrix class: dgRMatrix 


In [16]:
cellData <- adata$obs

# Print dimensions
cat("Matrix dimensions:\n")
cat("Rows (genes):", nrow(cellData), "\n")
cat("Columns (cells):", ncol(cellData), "\n")

# Print class
cat("Matrix class:", class(cellData), "\n")
# head(cellData)

featureData <- adata$var

# Print dimensions
cat("Matrix dimensions:\n")
cat("Rows (genes):", nrow(featureData), "\n")
cat("Columns (cells):", ncol(featureData), "\n")

# Print class
cat("Matrix class:", class(featureData), "\n")
# head(featureData)

Matrix dimensions:
Rows (genes): 15867 
Columns (cells): 40 
Matrix class: data.frame 
Matrix dimensions:
Rows (genes): 21412 
Columns (cells): 17 
Matrix class: data.frame 


In [17]:
# Create sparse expression set
sparse_eset <- createSparseEset(
  input_matrix = as(as(t(input_matrix), "matrix"), "dgCMatrix"),
  do.sparseConversion = TRUE,
  cellData = cellData,
  featureData = featureData,
  annotation = project_path,
  projectID = project_name,
  addMetaData = TRUE
)

sparse_eset

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 2.5 GiB”


Creating sparse eset from the input_matrix ...
	Adding meta data based on input_matrix ...
Done! The sparse eset has been generated: 21412 genes, 15867 cells.


SparseExpressionSet (storageMode: environment)
assayData: 21412 features, 15867 samples 
  element names: exprs 
protocolData: none
phenoData
  sampleNames: AAACCCAAGGTTACCT-1 AAACCCAAGTTGAAGT-1 ...
    TGTGTTGAGCCTATCT-1 (15867 total)
  varLabels: batch phase ... CellID (46 total)
  varMetadata: labelDescription
featureData
  featureNames: A1BG A1BG-AS1 ... ZZZ3 (21412 total)
  fvarLabels: mt ribo ... nCell (18 total)
  fvarMetadata: labelDescription
experimentData: use 'experimentData(object)'
Annotation: /nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/scMINER 

# network inference

In [22]:
## Columns with any illegal characters can not be used for groupping
outpath <- file.path(project_path, project_name, "SJARACNe")
cat("outpath:", outpath, "\n")

generateSJARACNeInput(
    input_eset = sparse_eset, 
    group_name = "cluster_str", 
    sjaracne_dir = outpath, 
    species_type = "hg", 
    driver_type = "TF_SIG", 
    downSample_N = 1000,
)

outpath: /nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/scMINER/test/SJARACNe 
No non-word characters (any character other than [a-zA-Z0-9_] ) were found in the group_name elements. group_name check passed...
Extracting TF drivers from the reference list...
A TF driver list of 2008 genes was generated!
Extracting SIG drivers from the reference list...
A SIG driver list of 9723 genes was generated!
5 groups were found from the input eset.
1/5: Generating SJARACNe inputs for group: C3
	Creating the output dirctory...
	3292 cells were found in this group...
	The metacell analysis by was skipped, since the superCell_N is null.
	1000 cells left after down-sampling...
	Writing gene expression matrix into /nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/scMINER/test/SJARACNe/C3/C3.20538_1000.exp.txt...
	Writing TF driver list into /nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/scMINER/test/SJARACNe/C3/TF/C3.1576_1000.tf.txt...
	Writing SIG driver list i

SparseExpressionSet (storageMode: environment)
assayData: 21412 features, 15867 samples 
  element names: exprs 
protocolData: none
phenoData
  sampleNames: AAACCCAAGGTTACCT-1 AAACCCAAGTTGAAGT-1 ...
    TGTGTTGAGCCTATCT-1 (15867 total)
  varLabels: batch phase ... CellID (46 total)
  varMetadata: labelDescription
featureData
  featureNames: A1BG A1BG-AS1 ... ZZZ3 (21412 total)
  fvarLabels: mt ribo ... nCell (18 total)
  fvarMetadata: labelDescription
experimentData: use 'experimentData(object)'
Annotation: /nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/scMINER 